# Animated comparison of saved Poincare sections

Run the two code cells below. **Play/Pause**, the cycle slider, and the 48 particle checkboxes control all three horizontal panels together. **Accumulate returns** builds the section progressively; disable it to show only the current return. Colors identify the same particle in every panel.

This visualization reads the three existing 48-particle, 5,000-cycle, 20-step/cycle studies. It performs no integration and makes no reference-accuracy claim. Each frame shows saved once-per-cycle returns, not the continuous motion between returns. Coordinates are periodic and normalized by cell length L, with identical [0, 1] axes. Playback advances 25 cycles every 100 ms by default; every accumulated return is retained.

Source calculations are read at float64 precision; only browser display coordinates are converted to float32. The generated HTML works offline and can also be opened directly. Run this notebook with the project Python environment and this notebook's directory as the working directory.


In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import IFrame, FileLink, display
from visualization.poincare_comparison import export_poincare_comparison

# Fixed source runs make the comparison reproducible; paths are relative to this notebook.
SOURCES = [('BM4', 'Poincare_BM4_48_radiales_5000_ciclos_20_steps_16_procesos_spot', 'aws_48p_5000c_20s_16proc_spot_20260920'), ('BM4Midpoint', 'Poincare_BM4Midpoint_48_radiales_5000_ciclos_20_steps_spot', 'aws_bm4midpoint_48p_5000c_20s_20260920'), ('RK4', 'Poincare_RK4_48_radiales_5000_ciclos_20_steps_local', 'local_rk4_48p_5000c_20s_20260920')]
CYCLES_PER_FRAME = 25
OUTPUT_HTML = Path("poincare_comparison.html")
particle_ids = np.arange(1, 49)
coordinates, titles, reference_metadata = [], [], None
for label, study, run_id in SOURCES:
    folder = Path("..") / study / "resultados" / run_id
    manifest = json.loads((folder / "COMPLETE.json").read_text())
    assert manifest["run_id"] == run_id
    for name in ("metadata.json", "positions_after_each_cycle.csv"):
        assert hashlib.sha256((folder / name).read_bytes()).hexdigest() == manifest["sha256"][name]
    metadata = json.loads((folder / "metadata.json").read_text())
    assert metadata["cycles"] == 5000 and metadata["steps_per_cycle"] == 20
    data = pd.read_csv(folder / "positions_after_each_cycle.csv", float_precision="round_trip")
    data = data.sort_values(["cycle", "particle"])
    np.testing.assert_array_equal(data["cycle"], np.repeat(np.arange(1, 5001), 48))
    np.testing.assert_array_equal(data["particle"], np.tile(particle_ids, 5000))
    colors = [metadata["colours"][str(p)] for p in particle_ids]
    np.testing.assert_array_equal(data["color"], np.tile(colors, 5000))
    if reference_metadata is None:
        reference_metadata = metadata
        reference_times = data[["time_normalized", "time_s"]].to_numpy()
        reference_radii = data["initial_radius_over_L"].to_numpy()
        reference_colors = colors
    else:
        np.testing.assert_allclose(data[["time_normalized", "time_s"]], reference_times, rtol=1e-13, atol=0)
        np.testing.assert_array_equal(data["initial_radius_over_L"], reference_radii)
        assert colors == reference_colors
    coordinates.append(data[["x_over_L", "y_over_L"]].to_numpy().reshape(5000, 48, 2))
    titles.append(f"{label} ({metadata['method']})" if label != metadata["method"] else label)
    print(f"{titles[-1]}: {len(data):,} returns loaded from {run_id}")
coordinates = np.stack(coordinates)  # (method, cycle, particle, x/y normalized by L).


BM4 (BM4Implicit): 240,000 returns loaded from aws_48p_5000c_20s_16proc_spot_20260920
BM4Midpoint: 240,000 returns loaded from aws_bm4midpoint_48p_5000c_20s_20260920
RK4: 240,000 returns loaded from local_rk4_48p_5000c_20s_20260920


In [2]:
export_poincare_comparison(
    OUTPUT_HTML, coordinates, particle_ids, reference_colors, titles,
    cycles_per_frame=CYCLES_PER_FRAME,
)
display(FileLink(str(OUTPUT_HTML)))
display(IFrame(str(OUTPUT_HTML), width="100%", height=1000))


/home/juan/Proyectos/GC2D_intranet/notebooks/developements/poincare_section/Poincare_comparacion_animada/poincare_comparison.html